# RNA 3D Structure Model Training Analysis

This notebook provides comprehensive analysis of production training runs for the RNA 3D structure prediction model.

In [ ]:
import os
import sys
import json
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import py3Dmol
from pathlib import Path

# Add parent directory to path for module imports
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Set plotting style
plt.style.use('ggplot')
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 12

## 1. Load Training Data

First, we'll load the training logs, GPU metrics, and validation results.

In [ ]:
def parse_training_logs(training_dir):
    """Parse training logs to extract metrics.
    
    Args:
        training_dir: Path to training directory containing logs
        
    Returns:
        DataFrame containing parsed training metrics
    """
    log_path = Path(training_dir) / "training_log.csv"
    if not log_path.exists():
        print(f"Warning: Training log not found at {log_path}")
        return pd.DataFrame()
    
    try:
        df = pd.read_csv(log_path)
        # Convert timestamp to datetime if it exists
        if 'timestamp' in df.columns:
            df['timestamp'] = pd.to_datetime(df['timestamp'])
        return df
    except Exception as e:
        print(f"Error parsing training logs: {e}")
        return pd.DataFrame()

def parse_gpu_metrics(training_dir):
    """Parse GPU monitoring metrics.
    
    Args:
        training_dir: Path to training directory containing GPU metrics
        
    Returns:
        DataFrame containing GPU metrics
    """
    gpu_metrics_path = Path(training_dir) / "gpu_metrics.csv"
    if not gpu_metrics_path.exists():
        print(f"Warning: GPU metrics not found at {gpu_metrics_path}")
        return pd.DataFrame()
    
    try:
        df = pd.read_csv(gpu_metrics_path)
        # Convert timestamp to datetime if it exists
        if 'timestamp' in df.columns:
            df['timestamp'] = pd.to_datetime(df['timestamp'])
        return df
    except Exception as e:
        print(f"Error parsing GPU metrics: {e}")
        return pd.DataFrame()

def parse_validation_results(training_dir):
    """Parse validation results.
    
    Args:
        training_dir: Path to training directory containing validation results
        
    Returns:
        DataFrame containing validation metrics
    """
    validation_path = Path(training_dir) / "validation_results.csv"
    if not validation_path.exists():
        print(f"Warning: Validation results not found at {validation_path}")
        return pd.DataFrame()
    
    try:
        df = pd.read_csv(validation_path)
        return df
    except Exception as e:
        print(f"Error parsing validation results: {e}")
        return pd.DataFrame()

In [ ]:
# Set the path to the training directory
# training_dir = "path/to/your/training/directory"
training_dir = "../logs/production_run_20250423"

# Load data
training_df = parse_training_logs(training_dir)
gpu_df = parse_gpu_metrics(training_dir)
validation_df = parse_validation_results(training_dir)

# Display basic information
print(f"Training data: {len(training_df)} records")
print(f"GPU metrics: {len(gpu_df)} records")
print(f"Validation results: {len(validation_df)} records")

# Show the first few rows of each dataset
if not training_df.empty:
    display(training_df.head())
if not gpu_df.empty:
    display(gpu_df.head())
if not validation_df.empty:
    display(validation_df.head())

## 2. Training Progress Analysis

Visualize how the model training progressed over time, including loss curves and learning rate adjustments.

In [ ]:
def plot_training_loss(training_df):
    """Plot training loss curves over epochs."""
    if training_df.empty:
        print("No training data available for plotting.")
        return
    
    # Check for required columns
    required_cols = ['epoch', 'total_loss', 'fape_loss', 'confidence_loss', 'angle_loss']
    missing_cols = [col for col in required_cols if col not in training_df.columns]
    
    if missing_cols:
        print(f"Warning: Missing columns in training data: {missing_cols}")
        available_loss_cols = [col for col in training_df.columns if 'loss' in col.lower()]
        if not available_loss_cols:
            print("No loss columns found in training data.")
            return
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 12))
    
    # Plot total loss
    if 'total_loss' in training_df.columns:
        axes[0].plot(training_df['epoch'], training_df['total_loss'], 'b-', label='Total Loss')
        axes[0].set_title('Total Training Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend()
        axes[0].grid(True)
    
    # Plot component losses
    component_losses = [col for col in training_df.columns if 'loss' in col.lower() and col != 'total_loss']
    for loss in component_losses:
        if loss in training_df.columns:
            axes[1].plot(training_df['epoch'], training_df[loss], label=loss.capitalize())
    
    axes[1].set_title('Component Losses')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

def plot_learning_rate(training_df):
    """Plot learning rate changes during training."""
    if 'learning_rate' not in training_df.columns:
        print("Learning rate data not available in training logs.")
        return
    
    plt.figure(figsize=(12, 6))
    plt.plot(training_df['epoch'], training_df['learning_rate'])
    plt.title('Learning Rate Schedule')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.yscale('log')
    plt.grid(True)
    plt.show()

In [ ]:
# Plot training loss curves
plot_training_loss(training_df)

# Plot learning rate changes if available
plot_learning_rate(training_df)

## 3. RMSD Analysis

Analyze the Root Mean Square Deviation (RMSD) distribution across validation samples.

In [ ]:
def plot_rmsd_distribution(validation_df):
    """Plot the distribution of RMSD values."""
    if validation_df.empty or 'rmsd' not in validation_df.columns:
        print("RMSD data not available in validation results.")
        return
    
    plt.figure(figsize=(12, 6))
    
    # Calculate histogram
    sns.histplot(validation_df['rmsd'], kde=True)
    
    # Add vertical lines for key statistics
    median_rmsd = validation_df['rmsd'].median()
    mean_rmsd = validation_df['rmsd'].mean()
    
    plt.axvline(median_rmsd, color='r', linestyle='--', label=f'Median: {median_rmsd:.2f}Å')
    plt.axvline(mean_rmsd, color='g', linestyle='--', label=f'Mean: {mean_rmsd:.2f}Å')
    
    plt.title('RMSD Distribution Across Validation Set')
    plt.xlabel('RMSD (Å)')
    plt.ylabel('Count')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # Print summary statistics
    print("RMSD Summary Statistics:")
    print(f"Mean: {mean_rmsd:.2f}Å")
    print(f"Median: {median_rmsd:.2f}Å")
    print(f"Min: {validation_df['rmsd'].min():.2f}Å")
    print(f"Max: {validation_df['rmsd'].max():.2f}Å")
    print(f"Std Dev: {validation_df['rmsd'].std():.2f}Å")

def plot_rmsd_by_length(validation_df):
    """Plot RMSD values against sequence length."""
    if validation_df.empty or 'rmsd' not in validation_df.columns or 'sequence_length' not in validation_df.columns:
        print("Required data (RMSD or sequence length) not available in validation results.")
        return
    
    plt.figure(figsize=(12, 6))
    
    # Create scatter plot
    plt.scatter(validation_df['sequence_length'], validation_df['rmsd'], alpha=0.6)
    
    # Add regression line
    z = np.polyfit(validation_df['sequence_length'], validation_df['rmsd'], 1)
    p = np.poly1d(z)
    plt.plot(validation_df['sequence_length'], p(validation_df['sequence_length']), 'r--', 
             label=f'Trend: y={z[0]:.4f}x+{z[1]:.4f}')
    
    plt.title('RMSD vs Sequence Length')
    plt.xlabel('Sequence Length (nucleotides)')
    plt.ylabel('RMSD (Å)')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Plot RMSD distribution
plot_rmsd_distribution(validation_df)

# Plot RMSD by sequence length
plot_rmsd_by_length(validation_df)

## 4. GPU Utilization Analysis

Analyze GPU resource utilization during training to identify potential bottlenecks.

In [ ]:
def plot_gpu_metrics(gpu_df):
    """Plot GPU utilization, memory usage, and temperature."""
    if gpu_df.empty:
        print("No GPU metrics available for plotting.")
        return
    
    # Check for required columns
    required_cols = ['timestamp', 'gpu_id', 'utilization', 'memory_used', 'memory_total', 'temperature']
    missing_cols = [col for col in required_cols if col not in gpu_df.columns]
    
    if missing_cols:
        print(f"Warning: Missing columns in GPU metrics: {missing_cols}")
        # Try to plot what we have
    
    # Convert timestamp to datetime if it's not already
    if 'timestamp' in gpu_df.columns and not pd.api.types.is_datetime64_any_dtype(gpu_df['timestamp']):
        gpu_df['timestamp'] = pd.to_datetime(gpu_df['timestamp'])
    
    # Set up plot
    fig, axes = plt.subplots(3, 1, figsize=(14, 15), sharex=True)
    
    # Get unique GPU IDs
    if 'gpu_id' in gpu_df.columns:
        gpu_ids = gpu_df['gpu_id'].unique()
    else:
        gpu_ids = [0]  # Default if no GPU ID column
    
    # Plot GPU utilization
    if 'utilization' in gpu_df.columns and 'timestamp' in gpu_df.columns:
        for gpu_id in gpu_ids:
            if 'gpu_id' in gpu_df.columns:
                gpu_data = gpu_df[gpu_df['gpu_id'] == gpu_id]
            else:
                gpu_data = gpu_df
            
            axes[0].plot(gpu_data['timestamp'], gpu_data['utilization'], label=f'GPU {gpu_id}')
        
        axes[0].set_title('GPU Utilization Over Time')
        axes[0].set_ylabel('Utilization (%)')
        axes[0].set_ylim(0, 105)
        axes[0].legend()
        axes[0].grid(True)
    
    # Plot memory usage
    if all(col in gpu_df.columns for col in ['memory_used', 'memory_total', 'timestamp']):
        for gpu_id in gpu_ids:
            if 'gpu_id' in gpu_df.columns:
                gpu_data = gpu_df[gpu_df['gpu_id'] == gpu_id]
            else:
                gpu_data = gpu_df
            
            # Convert to GB if in MB
            memory_factor = 1.0
            if gpu_data['memory_total'].max() > 1000:  # Likely in MB
                memory_factor = 1/1024.0  # Convert to GB
            
            axes[1].plot(gpu_data['timestamp'], 
                        gpu_data['memory_used'] * memory_factor, 
                        label=f'GPU {gpu_id} Used')
            
            # Plot total as horizontal line
            if len(gpu_data) > 0:
                total_mem = gpu_data['memory_total'].iloc[0] * memory_factor
                axes[1].axhline(y=total_mem, linestyle='--', 
                               color='r', label=f'GPU {gpu_id} Total ({total_mem:.1f} GB)')
        
        axes[1].set_title('GPU Memory Usage Over Time')
        axes[1].set_ylabel('Memory (GB)')
        axes[1].legend()
        axes[1].grid(True)
    
    # Plot temperature
    if 'temperature' in gpu_df.columns and 'timestamp' in gpu_df.columns:
        for gpu_id in gpu_ids:
            if 'gpu_id' in gpu_df.columns:
                gpu_data = gpu_df[gpu_df['gpu_id'] == gpu_id]
            else:
                gpu_data = gpu_df
            
            axes[2].plot(gpu_data['timestamp'], gpu_data['temperature'], label=f'GPU {gpu_id}')
        
        # Add warning threshold
        axes[2].axhline(y=80, linestyle='--', color='r', label='Warning Threshold')
        
        axes[2].set_title('GPU Temperature Over Time')
        axes[2].set_xlabel('Time')
        axes[2].set_ylabel('Temperature (°C)')
        axes[2].legend()
        axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot GPU utilization metrics
plot_gpu_metrics(gpu_df)

## 5. Structure Visualization

Visualize predicted 3D structures and compare with ground truth.

In [ ]:
def load_structure_data(training_dir, sample_id):
    """Load predicted and ground truth structure data for a specific sample.
    
    Args:
        training_dir: Path to training directory
        sample_id: ID of the sample to visualize
        
    Returns:
        Tuple of (predicted_coords, true_coords) as numpy arrays of shape (N, 3)
    """
    # This is a placeholder - actual implementation would depend on
    # how structure data is saved during training/validation
    
    predictions_dir = Path(training_dir) / "predictions"
    if not predictions_dir.exists():
        print(f"Predictions directory not found at {predictions_dir}")
        return None, None
    
    pred_file = predictions_dir / f"{sample_id}_predicted.npy"
    true_file = predictions_dir / f"{sample_id}_true.npy"
    
    if not pred_file.exists() or not true_file.exists():
        print(f"Structure files not found for sample {sample_id}")
        return None, None
    
    try:
        predicted_coords = np.load(pred_file)
        true_coords = np.load(true_file)
        return predicted_coords, true_coords
    except Exception as e:
        print(f"Error loading structure data: {e}")
        return None, None

def visualize_structure(coords, color='blue', view=None):
    """Visualize a 3D structure using py3Dmol.
    
    Args:
        coords: Numpy array of shape (N, 3) containing coordinates
        color: Color to use for visualization
        view: Existing py3Dmol view to add to (creates new one if None)
        
    Returns:
        py3Dmol view object
    """
    if coords is None:
        print("No coordinates available for visualization.")
        return None
    
    if view is None:
        view = py3Dmol.view(width=800, height=600)
    
    # Create a simple representation using spheres for each atom
    for i, coord in enumerate(coords):
        view.addSphere({
            'center': {'x': coord[0], 'y': coord[1], 'z': coord[2]},
            'radius': 0.5,
            'color': color
        })
    
    # Add bonds/lines between consecutive atoms
    for i in range(len(coords) - 1):
        view.addCylinder({
            'start': {'x': coords[i][0], 'y': coords[i][1], 'z': coords[i][2]},
            'end': {'x': coords[i+1][0], 'y': coords[i+1][1], 'z': coords[i+1][2]},
            'radius': 0.1,
            'color': color,
            'fromCap': True,
            'toCap': True
        })
    
    view.zoomTo()
    return view

def compare_structures(pred_coords, true_coords):
    """Compare predicted and ground truth structures side by side.
    
    Args:
        pred_coords: Numpy array of predicted coordinates
        true_coords: Numpy array of ground truth coordinates
    """
    if pred_coords is None or true_coords is None:
        print("Missing coordinates for comparison.")
        return
    
    # Create two separate views
    pred_view = visualize_structure(pred_coords, color='blue')
    true_view = visualize_structure(true_coords, color='green')
    
    # Display side by side
    display(pred_view, true_view)
    
    # Also create a combined view
    combined_view = py3Dmol.view(width=800, height=600)
    visualize_structure(pred_coords, color='blue', view=combined_view)
    visualize_structure(true_coords, color='green', view=combined_view)
    
    display(combined_view)

In [ ]:
# Visualize a sample structure
# sample_id = "example_id"  # Replace with an actual sample ID
# pred_coords, true_coords = load_structure_data(training_dir, sample_id)
# compare_structures(pred_coords, true_coords)

## 6. Checkpoint Analysis

Analyze multiple model checkpoints to identify the best performing model and potential ensemble strategies.

In [ ]:
def analyze_checkpoints(training_dir):
    """Analyze performance metrics across different model checkpoints.
    
    Args:
        training_dir: Path to training directory containing checkpoints
    """
    checkpoints_dir = Path(training_dir) / "checkpoints"
    if not checkpoints_dir.exists():
        print(f"Checkpoints directory not found at {checkpoints_dir}")
        return
    
    # Find checkpoint metric files
    metric_files = list(checkpoints_dir.glob("*_metrics.csv"))
    if not metric_files:
        print("No checkpoint metric files found.")
        return
    
    # Load and combine metrics
    checkpoint_metrics = []
    for metric_file in metric_files:
        try:
            # Extract checkpoint epoch/step from filename
            checkpoint_name = metric_file.stem.replace("_metrics", "")
            
            # Load metrics
            metrics_df = pd.read_csv(metric_file)
            
            # Add checkpoint identifier
            metrics_df['checkpoint'] = checkpoint_name
            
            checkpoint_metrics.append(metrics_df)
        except Exception as e:
            print(f"Error processing {metric_file}: {e}")
    
    if not checkpoint_metrics:
        print("Failed to load any checkpoint metrics.")
        return
    
    # Combine all metrics
    combined_metrics = pd.concat(checkpoint_metrics, ignore_index=True)
    
    # Plot RMSD improvement over checkpoints
    if 'rmsd' in combined_metrics.columns and 'checkpoint' in combined_metrics.columns:
        plt.figure(figsize=(12, 6))
        
        # Get average RMSD per checkpoint
        checkpoint_avg = combined_metrics.groupby('checkpoint')['rmsd'].mean().reset_index()
        checkpoint_avg['checkpoint_num'] = checkpoint_avg['checkpoint'].str.extract(r'(\d+)').astype(int)
        checkpoint_avg = checkpoint_avg.sort_values('checkpoint_num')
        
        plt.plot(checkpoint_avg['checkpoint'], checkpoint_avg['rmsd'], 'o-')
        plt.title('Average RMSD Across Model Checkpoints')
        plt.xlabel('Checkpoint')
        plt.ylabel('Average RMSD (Å)')
        plt.xticks(rotation=45)
        plt.grid(True)
        plt.tight_layout()
        plt.show()
        
        # Find best checkpoint
        best_checkpoint = checkpoint_avg.loc[checkpoint_avg['rmsd'].idxmin()]
        print(f"Best checkpoint: {best_checkpoint['checkpoint']} with RMSD {best_checkpoint['rmsd']:.2f}Å")
    else:
        print("RMSD or checkpoint column not found in metrics data.")

In [ ]:
# Analyze model checkpoints
analyze_checkpoints(training_dir)

## 7. Feature Importance Analysis

Analyze the importance of different input features through feature ablation studies.

In [ ]:
def visualize_feature_importance(training_dir):
    """Visualize feature importance from ablation studies.
    
    Args:
        training_dir: Path to training directory containing ablation results
    """
    ablation_file = Path(training_dir) / "ablation_results.csv"
    if not ablation_file.exists():
        print(f"Ablation results not found at {ablation_file}")
        return
    
    try:
        ablation_df = pd.read_csv(ablation_file)
        
        # Ensure we have the required columns
        if 'feature' not in ablation_df.columns or 'rmsd_change' not in ablation_df.columns:
            print("Required columns not found in ablation results.")
            return
        
        # Sort by impact
        ablation_df = ablation_df.sort_values('rmsd_change', ascending=False)
        
        # Plot bar chart
        plt.figure(figsize=(12, 8))
        
        # Create bar chart
        bars = plt.bar(ablation_df['feature'], ablation_df['rmsd_change'])
        
        # Color bars based on whether RMSD increased (red) or decreased (green)
        bar_colors = ['red' if x > 0 else 'green' for x in ablation_df['rmsd_change']]
        for bar, color in zip(bars, bar_colors):
            bar.set_color(color)
        
        plt.title('Feature Importance (Ablation Study)')
        plt.xlabel('Feature Category')
        plt.ylabel('RMSD Change (Å) When Feature Removed')
        plt.xticks(rotation=45, ha='right')
        plt.grid(True, axis='y')
        plt.tight_layout()
        plt.show()
        
        # Display table of numerical results
        display(ablation_df)
    except Exception as e:
        print(f"Error processing ablation results: {e}")

In [ ]:
# Visualize feature importance
# visualize_feature_importance(training_dir)

## 8. Training Performance Analysis

Analyze training speed, batch processing time, and bottlenecks.

In [ ]:
def analyze_training_performance(training_df):
    """Analyze batch processing times and training speed."""
    if training_df.empty:
        print("No training data available for performance analysis.")
        return
    
    # Check for required columns
    timing_cols = ['batch_time', 'data_loading_time', 'forward_time', 'backward_time']
    available_timing_cols = [col for col in timing_cols if col in training_df.columns]
    
    if not available_timing_cols:
        print("No timing data found in training logs.")
        return
    
    # Plot timing information
    plt.figure(figsize=(12, 6))
    
    for col in available_timing_cols:
        plt.plot(training_df['epoch'], training_df[col], label=col.replace('_', ' ').capitalize())
    
    plt.title('Training Performance Metrics')
    plt.xlabel('Epoch')
    plt.ylabel('Time (seconds)')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # Calculate and display average timing statistics
    print("Average timing statistics:")
    for col in available_timing_cols:
        print(f"{col.replace('_', ' ').capitalize()}: {training_df[col].mean():.4f} seconds")

In [ ]:
# Analyze training performance
analyze_training_performance(training_df)

## 9. Summary and Recommendations

Summarize key findings and provide recommendations for improving model performance.

In [ ]:
def generate_summary(training_df, validation_df, gpu_df):
    """Generate a summary of training results with recommendations."""
    print("# Training Run Summary")
    print("\n## Key Metrics")
    
    # Training details
    if not training_df.empty:
        print(f"- Total Epochs: {training_df['epoch'].max()}")
        
        if 'total_loss' in training_df.columns:
            final_loss = training_df.iloc[-1]['total_loss']
            best_loss = training_df['total_loss'].min()
            print(f"- Final Loss: {final_loss:.4f}")
            print(f"- Best Loss: {best_loss:.4f} (Epoch {training_df.loc[training_df['total_loss'].idxmin()]['epoch']})")
    
    # Validation results
    if not validation_df.empty and 'rmsd' in validation_df.columns:
        mean_rmsd = validation_df['rmsd'].mean()
        median_rmsd = validation_df['rmsd'].median()
        min_rmsd = validation_df['rmsd'].min()
        print(f"- Mean RMSD: {mean_rmsd:.2f}Å")
        print(f"- Median RMSD: {median_rmsd:.2f}Å")
        print(f"- Best RMSD: {min_rmsd:.2f}Å")
    
    # GPU utilization
    if not gpu_df.empty and 'utilization' in gpu_df.columns:
        mean_util = gpu_df['utilization'].mean()
        max_util = gpu_df['utilization'].max()
        print(f"- Mean GPU Utilization: {mean_util:.1f}%")
        print(f"- Peak GPU Utilization: {max_util:.1f}%")
    
    print("\n## Observations")
    
    # Loss curve analysis
    if not training_df.empty and 'total_loss' in training_df.columns:
        # Check if loss is still decreasing at the end
        last_epochs = training_df.tail(5)
        loss_slope = np.polyfit(last_epochs['epoch'], last_epochs['total_loss'], 1)[0]
        
        if loss_slope < -0.001:
            print("- Loss still decreasing at end of training - consider training for more epochs")
        elif loss_slope > 0.001:
            print("- Loss increasing at end of training - potential overfitting or learning rate issues")
        else:
            print("- Loss plateaued at end of training - model likely converged")
    
    # GPU utilization analysis
    if not gpu_df.empty and 'utilization' in gpu_df.columns:
        mean_util = gpu_df['utilization'].mean()
        if mean_util < 50:
            print("- Low GPU utilization - consider increasing batch size or model complexity")
        elif mean_util > 90:
            print("- Very high GPU utilization - model is computationally efficient")
    
    # RMSD distribution analysis
    if not validation_df.empty and 'rmsd' in validation_df.columns:
        # Check for outliers
        q75 = validation_df['rmsd'].quantile(0.75)
        q25 = validation_df['rmsd'].quantile(0.25)
        iqr = q75 - q25
        outlier_cutoff = q75 + 1.5 * iqr
        outliers = validation_df[validation_df['rmsd'] > outlier_cutoff]
        
        if len(outliers) > 0:
            pct_outliers = 100 * len(outliers) / len(validation_df)
            print(f"- {len(outliers)} samples ({pct_outliers:.1f}%) with unusually high RMSD values")
    
    print("\n## Recommendations")
    recommendations = []
    
    # Training recommendations
    if not training_df.empty:
        if 'total_loss' in training_df.columns:
            # Check if loss is still decreasing
            last_epochs = training_df.tail(5)
            loss_slope = np.polyfit(last_epochs['epoch'], last_epochs['total_loss'], 1)[0]
            
            if loss_slope < -0.001:
                recommendations.append("Continue training for more epochs to reach convergence")
            
            # Check for oscillating loss
            loss_diff = training_df['total_loss'].diff().abs()
            if loss_diff.mean() > 0.05 * training_df['total_loss'].mean():
                recommendations.append("Reduce learning rate to stabilize training")
    
    # GPU utilization recommendations
    if not gpu_df.empty and 'utilization' in gpu_df.columns and 'memory_used' in gpu_df.columns:
        mean_util = gpu_df['utilization'].mean()
        memory_usage_pct = 100 * gpu_df['memory_used'].mean() / gpu_df['memory_total'].iloc[0]
        
        if mean_util < 50 and memory_usage_pct < 80:
            recommendations.append("Increase batch size to improve GPU utilization")
        elif memory_usage_pct > 90:
            recommendations.append("Consider model optimization to reduce memory usage")
    
    # Add general recommendations
    recommendations.append("Implement checkpoint ensembling to improve prediction accuracy")
    recommendations.append("Consider analyzing specific RNA types or structural motifs separately")
    
    for rec in recommendations:
        print(f"- {rec}")

In [ ]:
# Generate summary and recommendations
generate_summary(training_df, validation_df, gpu_df)